# 05 · Alineación

Usa el **registro de predicciones** (`data/predictions_log/`, notebook 04) para:
1. mostrar todo mi roster con la predicción del modelo, la proyección de ESPN y el estado de lesión;
2. armar la alineación óptima con los slots de la liga y compararla con mi alineación actual;
3. sugerir los 5 mejores agentes libres por posición;
4. dar un **rango P10–P90** para cada jugador (regresión por cuantiles calibrada), sección 2;
5. calcular la **probabilidad de ganar** contra mi rival de esta semana y resolver las decisiones cerradas (sección 6);
6. mirar **varias semanas adelante**: puntos por semana, byes y huecos, fichajes a varias semanas y próximos rivales (sección 7).

**De dónde sale cada dato:**
- **Predicciones y proyecciones de ESPN:** del registro. Es la última predicción hecha antes del partido, así que son exactamente los números que quedaron guardados y fechados en git.
- **Roster, slots actuales, estado de lesión y agentes libres:** de ESPN **en vivo**, para decidir con la información más reciente.

**Umbral de 3 puntos:** el error típico del modelo es de unos 5–6 puntos por jugador y partido (notebook 03). Un cambio que gana menos de 3 puntos no es concluyente: los dos jugadores rinden prácticamente igual y la decisión puede depender de otros factores (clima, noticias de última hora, preferencia personal).



## 1. Datos

In [ ]:
import numpy as np
import polars as pl
import yaml

from fantasy_ml import data, espn, lineup as L, model as M, predictions_log as plog
from fantasy_ml.data import CONFIG, DATA_PROC

SEASON = 2026
THRESHOLD = 3.0  # puntos: por debajo, el cambio no es concluyente

league = espn.connect(SEASON)
WEEK = league.current_week
SLOTS = {k: v for k, v in league.settings.position_slot_counts.items() if v}

log_all = plog.read(SEASON)
log = plog.latest_pregame(log_all).filter(pl.col("week") == WEEK)
assert log.height, f"No hay predicciones registradas para la semana {WEEK}: ejecuta el notebook 04"
print(f"Semana {WEEK} · slots de la liga: {SLOTS}")
print(f"Predicciones del registro: {log.height} · generadas {log['generated_at_utc'].max():%Y-%m-%d %H:%M} UTC "
      f"· versión del código {log['code_version'].unique().to_list()}")

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(30)

## 1b. Rangos P10–P90: ¿están bien calibrados?

Además de la predicción puntual (la media), dos modelos LightGBM con **regresión por cuantiles** estiman los percentiles 10 y 90 de los puntos de cada jugador. Usan los **mismos hiperparámetros congelados** y las mismas features. Si los rangos están bien calibrados, el 80% de los resultados reales cae dentro, el 10% por debajo y el 10% por encima. Si los dos modelos se cruzaran (P10 > P90), el par se ordena.

Verificación con walk-forward semanal, igual que en el notebook 03 (cada semana se entrena solo con las anteriores):
- **2024:** mide la cobertura sin calibrar y **estima una calibración por posición**. Es un ajuste conformal: cada extremo se mueve lo justo para dejar un 10% en cada cola.
- **2025:** **verifica** esa calibración en una temporada que no se usó para estimarla.

La calibración se guarda en `config/ranges.yaml`, que usan las predicciones semanales (notebook 04). Pon `RECALIBRATE = True` para recalcular todo (~2 min).

In [ ]:
RECALIBRATE = False
PARAMS = data.load_config("model")["params"]
ranges_cfg_path = CONFIG / "ranges.yaml"
FEATURES = {"offense": "features_offense", "k": "features_k", "dst": "features_dst"}

def range_backtest(season):
    path = DATA_PROC / f"range_backtest_{season}.parquet"
    if RECALIBRATE or not path.exists():
        parts = []
        for g, f in FEATURES.items():
            df = pl.read_parquet(DATA_PROC / f"{f}.parquet")
            parts.append(M.walk_forward_ranges(df, g, PARAMS[g], M.eval_weeks(df, [season])).with_columns(group=pl.lit(g)))
        pl.concat(parts, how="diagonal_relaxed").write_parquet(path)
    return pl.read_parquet(path)

r24, r25 = range_backtest(2024), range_backtest(2025)
print(f"Cruces P10 > P90 antes de ordenar: 2024 {r24['crossed'].sum()} · 2025 {r25['crossed'].sum()} (de {r24.height + r25.height:,})")

if RECALIBRATE or not ranges_cfg_path.exists():
    cal = M.calibrate_ranges(r24)
    ranges_cfg = {"quantiles": list(M.QUANTILES), "calibrated_on": 2024, "verified_on": 2025,
                  "calibration": {k: list(v) for k, v in cal.items()}}
    with ranges_cfg_path.open("w", encoding="utf-8") as f:
        f.write("# Generado por notebooks/05_alineacion.ipynb (RECALIBRATE=True). Ajuste por posición (en puntos)\n"
                "# de los rangos P10–P90: [restar a P10, sumar a P90]. Estimado con 2024, verificado con 2025.\n")
        yaml.safe_dump(ranges_cfg, f, allow_unicode=True, sort_keys=False)
CAL = {k: tuple(v) for k, v in yaml.safe_load(ranges_cfg_path.read_text(encoding="utf-8"))["calibration"].items()}
print("Calibración (P10 −, P90 +):", CAL)

In [ ]:
def apply_cal(r):
    return r.with_columns(q10=pl.col("q10") - pl.col("position").replace_strict({k: v[0] for k, v in CAL.items()}),
                          q90=pl.col("q90") + pl.col("position").replace_strict({k: v[1] for k, v in CAL.items()}))

# jugadores relevantes para fantasy: mismo criterio que el backtest (notebook 03)
bt = (pl.read_parquet(DATA_PROC / "backtest_predictions.parquet")
        .with_columns(entity=pl.coalesce("player_id", "team")).select("season", "week", "entity", "relevant", "pred"))
rel = lambda r: r.with_columns(entity=pl.coalesce("player_id", "team")).join(bt, on=["season", "week", "entity"], how="left").filter("relevant")

cov = pl.concat([
    M.range_coverage(rel(r24), ["position"]).with_columns(temporada=pl.lit("2024 sin calibrar")),
    M.range_coverage(rel(r25), ["position"]).with_columns(temporada=pl.lit("2025 sin calibrar")),
    M.range_coverage(rel(apply_cal(r25)), ["position"]).with_columns(temporada=pl.lit("2025 calibrado ✓")),
])
cov.pivot(on="temporada", index="position", values="pct_dentro")

In [ ]:
print("2025 calibrado, jugadores relevantes (objetivo: 80 dentro, 10 debajo, 10 encima):")
M.range_coverage(rel(apply_cal(r25)), ["position"])

Con la calibración, la cobertura en 2025 queda cerca del 80% en todas las posiciones. Sin calibrar, K y D/ST cubrían solo ~74%. En RB los resultados que se salen del rango lo hacen más por arriba que por abajo, por los partidos explosivos. En la sección 2, P10 y P90 salen del **registro de predicciones**: se calculan antes de cada partido en el notebook 04, con esta misma calibración, así que al final de 2026 se podrá medir su cobertura real.

## 2. Mi roster

Un jugador **no está disponible** si ESPN lo marca OUT, IR o suspendido, o si no tiene predicción (semana libre o inactivo en nflverse). *Questionable* y *doubtful* se muestran con ⚠, pero siguen elegibles.

`p10`–`p90`: rango en el que el modelo espera que caigan sus puntos 8 de cada 10 veces (si juega). Un rango ancho indica más varianza: más techo y más riesgo.

In [ ]:
live = espn.rostered_projections(league, WEEK).filter("on_my_roster")
preds = log.select("espn_id", "pred_model", "pred_q10", "pred_q90", "espn_projection", "opponent", "kickoff_utc")

roster = (live.select("espn_id", name="espn_name", position="espn_position", my_slot="my_slot",
                      injury="espn_injury_status")
          .join(preds, on="espn_id", how="left")
          .with_columns(
              available=pl.col("pred_model").is_not_null() & ~pl.col("injury").fill_null("").is_in(list(L.UNAVAILABLE)),
              reason=pl.when(pl.col("pred_model").is_null()).then(pl.lit("sin predicción (bye/inactivo)"))
                       .when(pl.col("injury").is_in(list(L.UNAVAILABLE))).then(pl.col("injury"))
                       .otherwise(pl.lit("rendimiento")),
              aviso=pl.when(pl.col("injury").is_in(list(L.DOUBTFUL))).then(pl.lit("⚠ ") + pl.col("injury"))
                      .when(pl.col("injury").is_in(list(L.UNAVAILABLE))).then(pl.lit("✗ ") + pl.col("injury"))
                      .otherwise(pl.lit(""))))

SLOT_ORDER = {s: i for i, s in enumerate(["QB", "RB", "WR", "TE", "RB/WR/TE", "OP", "K", "D/ST", "BE", "IR"])}
(roster.with_columns(_o=pl.col("my_slot").replace_strict(SLOT_ORDER, default=99))
       .sort("_o", pl.col("pred_model"), descending=[False, True], nulls_last=True)
       .select(slot_actual="my_slot", jugador="name", pos="position", rival="opponent", lesion="injury", aviso="aviso",
               modelo=pl.col("pred_model").round(1), p10=pl.col("pred_q10").round(1), p90=pl.col("pred_q90").round(1),
               espn="espn_projection", modelo_menos_espn=(pl.col("pred_model") - pl.col("espn_projection")).round(1)))

## 3. Alineación óptima

Se asignan los mejores jugadores disponibles a cada slot: primero los fijos (QB, RB, WR, TE, K y D/ST) y después los FLEX (RB/WR/TE). Hago lo mismo con las proyecciones de ESPN para ver si ESPN recomendaría los mismos cambios.

In [ ]:
opt_model = L.optimal_lineup(roster, SLOTS, "pred_model")
opt_espn = L.optimal_lineup(roster.with_columns(available=pl.col("available") & pl.col("espn_projection").is_not_null()),
                            SLOTS, "espn_projection")

show = roster.select("espn_id", "name", "position", "my_slot", "pred_model", "espn_projection", "aviso")
lineup_table = (opt_model.with_row_index("_i")
    .join(show, on="espn_id", how="left")
    .join(opt_espn.select("espn_id", espn_tambien=pl.lit("✓")), on="espn_id", how="left")
    .sort("_i")
    .select(slot="slot", jugador="name", pos="position", slot_actual="my_slot", aviso="aviso",
            modelo=pl.col("pred_model").round(1), espn="espn_projection", espn_lo_alinea=pl.col("espn_tambien").fill_null("✗")))

cur = roster.filter(~pl.col("my_slot").is_in(list(L.BENCH_SLOTS)))
print(f"Puntos proyectados por el modelo · alineación actual: {cur['pred_model'].fill_null(0).sum():.1f} "
      f"· óptima: {lineup_table['modelo'].fill_null(0).sum():.1f}")
lineup_table

## 4. Cambios recomendados

Solo cuenta **quién es titular**, no en qué slot: mover a un RB del slot RB al FLEX no es un cambio. Cada jugador que entra se empareja con uno que sale de su misma posición, o con el peor titular si el cambio pasa por el FLEX.

- **Concluyente:** el modelo gana 3 o más puntos, o el que sale no puede jugar (OUT/IR/bye).
- **No concluyente:** gana menos de 3 puntos. Los dos jugadores rinden prácticamente igual.

In [ ]:
changes = L.lineup_changes(roster, opt_model, "pred_model", THRESHOLD)
espn_starters = set(opt_espn["espn_id"].drop_nulls().to_list())
name_to_id = dict(zip(roster["name"], roster["espn_id"]))

if changes.is_empty():
    print("✓ Tu alineación actual ya es la óptima según el modelo.")
else:
    changes = changes.with_columns(
        veredicto=pl.when("concluyente").then(pl.lit("✓ hacer el cambio"))
                    .otherwise(pl.lit(f"≈ no concluyente (< {THRESHOLD:g} pts)")),
        espn_de_acuerdo=pl.col("entra").map_elements(lambda n: "sí" if name_to_id[n] in espn_starters else "no",
                                                      return_dtype=pl.Utf8))
    display(changes.select("veredicto", "entra", "pos_entra", "sale", "pos_sale", "motivo_salida",
                           "pred_entra", "pred_sale", "diferencia", "espn_de_acuerdo"))

## 5. Mejores agentes libres por posición

Top 5 por predicción del modelo entre los agentes libres de ESPN **en este momento** que tienen predicción en el registro. Los que no la tienen no aparecen en los rosters activos de nflverse (lesionados, practice squad, etc.).

**`mejora_alineacion`:** cuántos puntos sube mi alineación óptima si agrego a ese jugador. Es 0 si no entraría de titular. Es mejor que compararlo con el peor jugador de su posición, porque tiene en cuenta los FLEX: un WR nuevo puede desplazar al RB más débil del FLEX, no al WR de la banca. Se aplica el mismo umbral de 3 puntos.

In [ ]:
fa = (espn.free_agent_projections(league, WEEK)
      .select("espn_id", name="espn_name", position="espn_position", injury="espn_injury_status")
      .join(preds, on="espn_id", how="inner")
      .filter(~pl.col("injury").fill_null("").is_in(list(L.UNAVAILABLE))))

top_fa = fa.with_columns(available=pl.lit(True)).sort("pred_model", descending=True).group_by("position", maintain_order=True).head(5)
top_fa = (top_fa.with_columns(L.pickup_gain(roster, top_fa, SLOTS, "pred_model").round(1))
            .with_columns(_o=pl.col("position").replace_strict(SLOT_ORDER, default=99),
                          veredicto=pl.when(pl.col("mejora_alineacion") >= THRESHOLD).then(pl.lit("✓ vale la pena"))
                                      .when(pl.col("mejora_alineacion") > 0).then(pl.lit(f"≈ no concluyente (< {THRESHOLD:g} pts)"))
                                      .otherwise(pl.lit("no entraría de titular")))
            .sort("_o", pl.col("pred_model"), descending=[False, True])
            .select(pos="position", jugador="name", rival="opponent", lesion="injury",
                    modelo=pl.col("pred_model").round(1), espn="espn_projection", mejora_alineacion="mejora_alineacion",
                    veredicto="veredicto"))
print(f"Agentes libres con predicción: {fa.height}")
top_fa

## 6. Probabilidad de ganar contra mi rival

Código: `src/fantasy_ml/matchup.py`. Se ejecuta también automáticamente cada domingo y queda registrada en `data/predictions_log/winprob_<temporada>.csv`.

- **Alineaciones:** la mía es la **actual de ESPN**, y la óptima se muestra como alternativa. La del rival es su alineación actual con los huecos cubiertos: un slot vacío o un titular OUT, IR, suspendido, doubtful, en bye o sin predicción se llena con **su mejor suplente disponible**.
- **Proyecciones:** las del registro de predicciones, es decir, la media y el rango P10–P90 registrados antes del partido.
- **Monte Carlo (10,000 simulaciones):**
  1. **Quién juega:** cada jugador juega según su estado de ESPN (questionable 75%, doubtful 25%).
  2. **Cuántos puntos hace si juega:** se muestrean de los **residuos reales del backtest de 2025**, según posición y nivel de predicción, **escalados para que su dispersión coincida con su propio rango P10–P90** y centrados en su predicción.
  3. **Si un titular no juega:** entra el mejor suplente disponible de su banca, como haría el manager antes del partido.
  4. **Partidos ya terminados** (por ejemplo, el del jueves si se ejecuta el domingo): se usan los puntos reales de ESPN.
- **P(ganar)** = P(mis puntos > los del rival) + ½·P(empate). Se compara con la de ESPN.

⚠ **Limitación:** los jugadores se simulan de forma independiente. En la realidad un QB y sus receptores suben o bajan juntos, y a una D/ST le va peor si el QB rival tiene un buen día. Por eso las probabilidades salen algo más extremas de lo que deberían.

In [ ]:
from fantasy_ml import matchup as MU

res = MU.analyze(n_sims=10000)
a, o = res["actual"], res["optimal"]
print(f"Semana {res['week']} vs {res['info']['rival']}")
pl.DataFrame([
    {"alineación": "actual (ESPN)", "p_ganar": round(a["p_win"], 3), "mis_puntos": round(a["exp_me"], 1),
     "mi_rango_80": f"{a['me_p10']:.0f}–{a['me_p90']:.0f}", "rival_puntos": round(a["exp_rival"], 1),
     "rival_rango_80": f"{a['rival_p10']:.0f}–{a['rival_p90']:.0f}"},
    {"alineación": "óptima (modelo)", "p_ganar": round(o["p_win"], 3), "mis_puntos": round(o["exp_me"], 1),
     "mi_rango_80": f"{o['me_p10']:.0f}–{o['me_p90']:.0f}", "rival_puntos": round(o["exp_rival"], 1),
     "rival_rango_80": f"{o['rival_p10']:.0f}–{o['rival_p90']:.0f}"},
    {"alineación": "ESPN", "p_ganar": res["espn_p_win"], "mis_puntos": res["info"]["espn_proj_me"],
     "mi_rango_80": None, "rival_puntos": res["info"]["espn_proj_rival"], "rival_rango_80": None},
])

### Alineación del rival (huecos cubiertos) y mis cambios para llegar a la óptima

In [ ]:
(res["rival"].filter(pl.col("starter_slot").is_not_null())
    .select("starter_slot", slot_en_espn="slot", jugador="name", lesion="injury", pred=pl.col("pred_model").round(1),
            p10=pl.col("pred_q10").round(1), p90=pl.col("pred_q90").round(1), p_jugar="p_play"))

In [ ]:
cur = set(res["mine_current"].filter(pl.col("starter_slot").is_not_null())["name"])
opt = set(res["mine_optimal"].filter(pl.col("starter_slot").is_not_null())["name"])
print("Entran en la óptima:", sorted(opt - cur) or "—", "· salen:", sorted(cur - opt) or "—")

### Decisiones cerradas: ¿menos o más varianza?

Para cada titular mío, los suplentes elegibles a **menos de 3 puntos esperados**:
- **`si_favorito`:** el de menos varianza (asegura el resultado).
- **`si_no_favorito`:** el de más varianza (hace falta un partido grande).
- **`conviene_hoy`:** la opción que da **más probabilidad de ganar esta semana**, calculada con los mismos sorteos.
- **`decide`:** si esa elección la marcó la media o la varianza.

La desviación incluye la posibilidad de que el jugador no juegue.

In [ ]:
print(f"P(ganar) con la alineación actual: {a['p_win']:.1%} → {'favorito' if a['p_win'] >= 0.5 else 'no favorito'}")
res["decisions"]

### ¿La dispersión simulada respeta los rangos?

Para cada jugador que aún no ha jugado, comparo los percentiles 10 y 90 simulados (si juega) con su rango P10–P90 del registro. Deberían coincidir en ancho, y la media simulada con la predicción.

In [ ]:
rc = MU.range_consistency(res["players"], res["sims"])
print(f"{rc.height} jugadores · ancho medio del rango {rc['ancho_rango'].mean():.2f} vs simulado {rc['ancho_sim'].mean():.2f} · "
      f"|P10 sim − P10| medio {(rc['sim_p10'] - rc['q10']).abs().mean():.2f} · |P90 sim − P90| medio {(rc['sim_p90'] - rc['q90']).abs().mean():.2f} · "
      f"|media sim − predicción| máx {(rc['sim_media'] - rc['pred']).abs().max():.2f}")
rc.select("name", "position", pl.col("pred", "q10", "q90", "sim_p10", "sim_p90").round(1)).head(10)

## 7. Próximas semanas

Código: `src/fantasy_ml/outlook.py`. Reutiliza el contexto del analizador de trades (notebook 06): proyección semana a semana con byes y probabilidad de jugar, agentes libres y Monte Carlo de 2,000 temporadas. Horizonte: las próximas **4 semanas**; los byes y huecos, toda la temporada.

**Una diferencia con el analizador de trades:** aquí los agentes libres **solo cubren los slots que tu roster no puede llenar** (bye, lesión), porque se describe *tu* roster. En los trades, en cambio, compiten por todos los slots (nivel de reemplazo). Los fichajes (7.3) comparan tu roster con y sin el agente libre, que es la acción real de fichar.

In [ ]:
from fantasy_ml import outlook as O, trades as T

ctx = T.build_trade_context(2026)
out = O.analyze(ctx, horizon=4)
print(f"Semanas analizadas: {out['horizon']}")

### 7.1 Mis puntos por semana

Alineación óptima de cada semana con tu roster actual (byes, lesiones y huecos cubiertos por agentes libres). Rango del 80% simulado.

In [ ]:
out["weekly"].with_columns(pl.col("esperado", "p10", "p90").round(1))

### 7.2 Byes y huecos

`T` titular en la óptima · `B` banca · `bye` · `fuera` (OUT/IR). Debajo, los **huecos reales** de toda la temporada: slots que tu roster no puede llenar esa semana y el mejor agente libre disponible hoy para cubrirlos.

In [ ]:
out["calendar"]

In [ ]:
out["needs"] if out["needs"].height else print("Tu roster cubre todos los slots en lo que queda de temporada.")

### 7.3 Agentes libres a varias semanas

Cuánto suben tus puntos esperados **en las próximas 4 semanas** si fichas a cada agente libre (los mejores 6 de cada posición en ese horizonte). Si tu roster está lleno, se suelta al jugador cuya salida conviene más (`suelto`). `cubre_byes` indica las semanas en que entraría de titular mientras uno de tus jugadores descansa.

In [ ]:
out["pickups"].filter(pl.col("ganancia") > 0).head(12)

### 7.4 Próximos rivales

Probabilidad de ganar contra el rival que te toca cada semana según el calendario de la liga, con la alineación óptima de los dos equipos y los mismos sorteos del Monte Carlo. Para la semana actual, el cálculo de la sección 6 es más preciso: usa tu alineación real y la escala de cada jugador según su rango P10–P90.

In [ ]:
out["matchups"]